# L29 · 常见概率分布（Probability Distribution） — 均匀（Uniform）、正态（Gaussian）、伯努利（Bernoulli）：PDF / CDF 与采样（Sampling）

**目标**：实现 `gaussian_pdf`，验证正态分布的 68–95–99.7% 经验法则，用 `rng.binomial` 采样 Bernoulli 分布，用梯形数值积分（`np.trapezoid`）计算 CDF。

**为什么对 Aurora 重要**：正态分布是 He 权重初始化（`np.random.normal(0, scale, shape)`）的理论基础；Bernoulli 分布是 dropout 的概率模型（每个神经元以概率 p 独立"关闭"）；CDF 对应特征分布的分位数分析，常用于训练数据质量诊断。

## 实验入口：采样如何收敛到分布形状

生成 N=10 和 N=10000 个样本（Sample），观察直方图的轮廓：小样本时形状粗糙，大样本时逼近理论 PDF 曲线。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
u = rng.uniform(-3, 3, 5000)    # 均匀: 区间内处处等可能
g = rng.normal(0, 1, 5000)      # 正态: 中间多、两边少
fig, ax = plt.subplots(1, 2, figsize=(9,3))
ax[0].hist(u, bins=40); ax[0].set_title('uniform')
ax[1].hist(g, bins=40); ax[1].set_title('normal'); plt.show()

## 动手观察：随机代码要多试几次

多次运行下面的格，观察小样本（N=10）的均值（Mean）每次波动多大，大样本（N=10000）的均值有多稳定。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    u = rng.uniform(0, 1, size=n)
    print(f'n={n:6d}  均匀分布样本均值={np.mean(u):.3f}  '  # 理论值 0.5
          f'std={np.std(u):.3f}')           # 理论值 1/sqrt(12)≈0.289


## 代码实验：多次重复同一个随机实验

这个实验展示样本量越大、重复实验之间均值的方差越小——这是大数定律（Law of Large Numbers，LLN）的直接体现。

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        samples = rng.normal(0, 1, size=n)  # 标准正态
        estimates.append(np.mean(samples))
    print(f'n={n:4d} -> 样本均值 5次:', np.round(estimates, 3),
          '平均=', round(float(np.mean(estimates)), 3))


## 2. ✏️ 实现正态分布概率密度 `gaussian_pdf(x, mu, sigma)`

`pdf = exp(−(x−μ)² / (2σ²)) / (σ·√(2π))`

**推理路线**：
1. 计算指数项：`exponent = -((x - mu)**2) / (2 * sigma**2)`。分子是偏离均值的平方（总为非负），`sigma**2` 控制衰减速度——sigma 越大衰减越慢。
2. 计算归一化（Normalization）系数：`norm = sigma * np.sqrt(2 * np.pi)`。这个分母保证整条曲线积分恰好为 1，无论 sigma 取何值。
3. 返回 `np.exp(exponent) / norm`。NumPy 自动广播：x 是数组时逐元素计算，是标量时返回标量。

**参考输入输出**：`mu=0, sigma=1, x=0` → pdf=1/√(2π)≈0.399；`x=1` → pdf≈0.242（下降到峰值的 60.7%）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


**提示**：用 `np.exp`、`np.sqrt`、`np.pi`。

写 `gaussian_pdf` 前明确三件事：
- 输入：`x`（评估点，数组），`mu`（均值），`sigma`（标准差（Standard Deviation，SD），须 > 0）
- 关键步骤：计算指数项 `exp(-(x-mu)**2 / (2*sigma**2))`，再除以归一化系数 `sigma * sqrt(2π)`
- 返回：与 `x` 形状相同的 PDF 值数组，所有值均非负

In [ ]:
def gaussian_pdf(x, mu=0.0, sigma=1.0):
    # ✏️ TODO: 返回正态分布在 x 处的概率密度
    return None  # ← 改这里


In [ ]:
import numpy as np
xs = np.linspace(-4, 4, 9)
vals = gaussian_pdf(xs, 0.0, 1.0)
peak = gaussian_pdf(np.array([0.0]))
if peak is None:
    print('⬜ 请先实现 gaussian_pdf，再运行此格')
else:
    peak_val = peak[0]
    assert abs(peak_val - 1/np.sqrt(2*np.pi)) < 1e-9, '峰值应在 μ 处'
    assert abs(gaussian_pdf(np.array([1.0]))[0] - gaussian_pdf(np.array([-1.0]))[0]) < 1e-12, '应关于 μ 对称'
    print('✅ 通过：高斯密度峰值=', round(peak_val, 4))


**🔗 Aurora 连接**：`gaussian_pdf` 是 Aurora 生成流水线的密度评估原语。He 权重初始化（`np.random.normal(0, std, shape)`）、VAE 隐变量采样（`z ~ N(0, I)`）、扩散模型加噪（逐步叠加 `N(0, β_t·I)` 噪声）——三处都直接对应今天手写的公式。


In [ ]:
import numpy as np

# 固定 mu=0，改变 sigma，观察峰值和宽度的变化
xs = np.linspace(-4, 4, 100)
for sigma in [0.5, 1.0, 2.0]:
    vals = gaussian_pdf(xs, mu=0.0, sigma=sigma)
    if vals is None:
        print('⬜ 请先实现 gaussian_pdf')
        break
    print(f'sigma={sigma:.1f}  峰值={gaussian_pdf(np.array([0.0]), 0.0, sigma)[0]:.4f}')
    print(f'       曲线下面积≈{np.trapezoid(vals, xs):.4f}  （应≈1.0）')


## 参数实验：只改一个旋钮

固定 `mu=0`，把 `sigma` 从 0.5 改到 2，观察峰值从 0.8 降到 0.2（更扁平），但曲线下面积始终为 1。这说明 sigma 控制"集中程度"，不改变总概率。再把 `mu` 从 0 改到 2，曲线整体平移但形状（峰值、宽度）不变——mu 只决定分布的中心位置。

In [ ]:
import numpy as np

# 正态分布的 CDF：用数值积分近似
xs = np.linspace(-4, 4, 1000)
for sigma in [0.5, 1.0, 2.0]:
    pdf = gaussian_pdf(xs, 0.0, sigma)
    if pdf is None:
        print('⬜ 请先实现 gaussian_pdf')
        break
    cdf = np.cumsum(pdf) * (xs[1] - xs[0])
    # 经验法则：μ±1σ 覆盖 ≈68% 的面积
    mask = np.abs(xs) <= sigma
    area_1sigma = np.trapezoid(pdf[mask], xs[mask])
    print(f'sigma={sigma:.1f}  μ±1σ 面积={area_1sigma:.3f}  (期望≈0.683)')


## 3. 伯努利分布（Bernoulli Distribution）

单次二值试验：以概率 `p` 成功（X=1），概率 `1-p` 失败（X=0）。

**PMF（概率质量函数，Probability Mass Function）**：
```
P(X=k) = p^k * (1-p)^(1-k),  k ∈ {0,1}
```

**期望（Expectation）** = `p`，**方差（Variance）** = `p(1-p)`。

**Aurora 连接**：dropout 掩码的每个元素服从 Bernoulli(p=1-dropout_rate)，独立地决定是否保留该神经元。音频增强（data augmentation）中的 SpecAugment 也用 Bernoulli 掩码随机遮挡频率段。

In [ ]:
import numpy as np

# 伯努利 PMF
def bernoulli_pmf(k, p):
    k = np.asarray(k)
    return (p ** k) * ((1 - p) ** (1 - k))

# 验证 PMF：k=0 和 k=1 的概率之和为 1
for p in [0.3, 0.5, 0.8]:
    pmf0 = bernoulli_pmf(0, p)
    pmf1 = bernoulli_pmf(1, p)
    print(f'p={p}: P(X=0)={pmf0:.3f}, P(X=1)={pmf1:.3f}, sum={pmf0+pmf1:.3f}')
    assert abs(pmf0 + pmf1 - 1.0) < 1e-9

# 采样验证：大样本均值趋近 p
rng = np.random.default_rng(0)
for p in [0.3, 0.5, 0.8]:
    samples = rng.binomial(1, p, size=100_000)
    print(f'p={p}: 样本均值={samples.mean():.4f}  方差={samples.var():.4f}  理论方差={p*(1-p):.4f}')

print('✅ 伯努利分布 PMF 与采样均值验证通过')


## 4. 累积分布函数（Cumulative Distribution Function，CDF）

CDF 定义：`F(x) = P(X ≤ x)` — 随机变量取值不超过 x 的概率。

| 分布 | CDF 形式 |
|---|---|
| 均匀 U[a,b] | F(x) = (x-a)/(b-a) 对 x∈[a,b] |
| 正态 N(μ,σ²) | 用数值积分：∫ pdf(t)dt |
| 伯努利 B(p) | F(0)=1-p，F(1)=1 |

**性质**：CDF 单调非降，F(-∞)=0，F(+∞)=1；连续分布的 CDF 处处连续。

**用途**：给定 CDF，对任意区间 [a,b] 的概率直接相减：`P(a < X ≤ b) = F(b) - F(a)`。

In [ ]:
import numpy as np

# 正态分布 CDF（数值积分：梯形法则）
def normal_cdf(x_eval, mu=0.0, sigma=1.0, n=2000):
    xs = np.linspace(mu - 6*sigma, x_eval, n)
    pdf = np.exp(-0.5*((xs - mu)/sigma)**2) / (sigma * np.sqrt(2*np.pi))
    return np.trapezoid(pdf, xs)

# 验证经验法则：N(0,1) 在 [-1,1] 内概率 ≈ 68.27%
p_within_1sd = normal_cdf(1.0) - normal_cdf(-1.0)
p_within_2sd = normal_cdf(2.0) - normal_cdf(-2.0)
p_within_3sd = normal_cdf(3.0) - normal_cdf(-3.0)

print(f'P(-1 < X < 1) = {p_within_1sd:.4f}  (理论 68.27%)')
print(f'P(-2 < X < 2) = {p_within_2sd:.4f}  (理论 95.45%)')
print(f'P(-3 < X < 3) = {p_within_3sd:.4f}  (理论 99.73%)')

assert abs(p_within_1sd - 0.6827) < 0.002
assert abs(p_within_2sd - 0.9545) < 0.002
assert abs(p_within_3sd - 0.9973) < 0.002
print('✅ 正态分布 CDF 经验法则验证通过')


## 本课收束

现在可以用 `gaussian_pdf(x, mu, sigma)` 在任意点求正态分布的密度值，并通过 `np.random.uniform` / `np.random.normal` 生成两种分布的样本。`gaussian_pdf` 对应 Aurora 中加噪步骤的密度评估和 VAE 的 KL 散度计算。下一节用 softmax 把未归一化得分变成概率分布，今天手写的归一化系数思路在那里会再出现。

In [ ]:
# 小检查：用 gaussian_pdf 验证经验法则
# 正态分布 μ±1σ 覆盖约 68.27% 的面积
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.normal(0, 1, size=n)
    within_1std = np.sum(np.abs(samples) <= 1.0) / n
    print(f'n={n:4d} -> μ±1σ 内的样本比例={within_1std:.3f}  (理论≈0.683)')
